In [ ]:
ingest_date = None

In [ ]:
%run 00_config

In [ ]:
from pyspark.sql import functions as F

def write(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)

patient = spark.read.table("silver_patient").where("is_current = true")
write(patient.select(
        F.col("business_id").alias("patient_id"),
        F.concat_ws(" ", "name_given", "name_family").alias("name_full"),
        "gender", F.to_date("birth_date").alias("birth_date"), "marital_status", "deceased")
     .withColumn("age", F.floor(F.datediff(F.current_date(), F.col("birth_date")) / 365.25).cast("int")),
     "gold_dim_patient")

enc = spark.read.table("silver_encounter").where("is_current = true")
write(enc.select(
        F.col("business_id").alias("encounter_id"), "patient_id", "status", "class_code", "type_text",
        F.to_timestamp("period_start").alias("period_start"),
        F.to_timestamp("period_end").alias("period_end"),
        F.to_date("period_start").alias("encounter_date")), "gold_fact_encounter")

obs = spark.read.table("silver_observation").where("is_current = true")
write(obs.select(
        F.col("business_id").alias("observation_id"), "patient_id", "encounter_id", "code_text",
        F.col("value_quantity").cast("double").alias("value_quantity"), "value_unit", "value_string",
        F.to_timestamp("effective_datetime").alias("effective_datetime")), "gold_fact_observation")

cond = spark.read.table("silver_condition").where("is_current = true")
write(cond.select(
        F.col("business_id").alias("condition_id"), "patient_id", "encounter_id", "code_text",
        "clinical_status", "verification_status",
        F.to_timestamp("onset_datetime").alias("onset_datetime"),
        F.to_date("recorded_date").alias("recorded_date")), "gold_fact_condition")

for t in ["gold_dim_patient", "gold_fact_encounter", "gold_fact_observation", "gold_fact_condition"]:
    print(t, spark.read.table(t).count())

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW gold_vw_patient_360 AS
SELECT p.patient_id, p.name_full, p.gender, p.age,
       COUNT(DISTINCT e.encounter_id)   AS encounter_count,
       COUNT(DISTINCT o.observation_id) AS observation_count,
       COUNT(DISTINCT c.condition_id)   AS condition_count
FROM gold_dim_patient p
LEFT JOIN gold_fact_encounter   e ON e.patient_id = p.patient_id
LEFT JOIN gold_fact_observation o ON o.patient_id = p.patient_id
LEFT JOIN gold_fact_condition   c ON c.patient_id = p.patient_id
GROUP BY p.patient_id, p.name_full, p.gender, p.age
""")
spark.sql("""
CREATE OR REPLACE VIEW gold_vw_observation_enriched AS
SELECT o.observation_id, o.patient_id, p.name_full, p.gender,
       o.encounter_id, o.code_text, o.value_quantity, o.value_unit, o.effective_datetime
FROM gold_fact_observation o
LEFT JOIN gold_dim_patient p ON p.patient_id = o.patient_id
""")
display(spark.sql("SELECT * FROM gold_vw_patient_360 ORDER BY encounter_count DESC LIMIT 15"))